# Exp10.2.2 — Valid-Length vs Whole-Window Hidden-State Probes

Aggregation-only notebook. It reads finalized Exp10.2.2 CSV/JSON artifacts and performs no checkpoint inference or training.

In [ ]:
from pathlib import Path
import json
import pandas as pd

def find_repo_root(start=Path.cwd()):
    for p in (start, *start.parents):
        if (p / 'scripts').exists() and (p / 'notebooks').exists():
            return p
    raise RuntimeError('Repository root not found')

repo = find_repo_root()
root = repo / 'notebooks' / 'artifacts' / 'experiment_10_2_2_valid_window_probes' / 'exp10_2_1_checkpoints_valid_window_v1'
manifest = json.loads((root / 'manifest.json').read_text())
manifest


## Primary support comparison

Every contrast is `whole-window - valid-length` for the same frozen Exp10.2.1 checkpoint and the same probe definition.

In [ ]:
summary = pd.read_csv(root / 'support_contrast_summary.csv')
display(summary)


## Key L1/L2 probes

Focus on pre-reset Fixed250 mean and communication Fixed250 count for both layers.

In [ ]:
key = pd.read_csv(root / 'key_probe_summary.csv')
display(key)


## BA and Macro-F1 side by side

In [ ]:
runs = pd.read_csv(root / 'support_contrasts.csv')
cols = [
    'coding','l2_mem_shift','seed','layer','state','aggregation',
    'test_balanced_accuracy_valid','test_balanced_accuracy_window','delta_test_balanced_accuracy',
    'test_macro_f1_valid','test_macro_f1_window','delta_test_macro_f1'
]
display(runs[cols].sort_values(['coding','l2_mem_shift','layer','state','aggregation','seed']))


## Full probe table

In [ ]:
probe_summary = pd.read_csv(root / 'probe_summary.csv')
display(probe_summary)


## Interpretation guide

- Positive `window-valid` BA/F1: post-valid residual dynamics add linearly decodable class evidence under that probe.
- Near zero: padded-tail dynamics add little beyond the valid region.
- Negative: including residual dynamics dilutes or distorts the representation.
- Increasing support gain with L2 tau: evidence that longer L2 memory produces a longer class-dependent residual tail.
- Large Weighted31 support gain: residual burst dynamics persist after the valid input; small gain instead points to valid-period burst/scale effects as the primary Weighted31 issue.